# 002 — Formal Track: Exact Rational Algebra, Simplex Tableaux, and Duality Certificates
## Linear Programming Duality, Zero-Float Arithmetic, Simplicial Physics, and Formal Verification

**Paper Package 002** — *L1-Native Cohomology of Finite Defect Fields*  
**Audience:** Research mathematicians, theoretical physicists, and formal verification specialists.  
**AuthorityEffect:** `None`

---

### Abstract & Mathematical Architecture
This notebook provides the rigorous mathematical, algorithmic, and formal verification foundation for Paper 002. Rational simplex, feasibility, stationarity, topology and PSD decisions are exact. Continuous-$p$ iteration, singular values and irrational complex moduli use binary64 diagnostics; floating-point displays are not exact certificates.

The notebook demonstrates:
1. **The Zero-Float Law & Exact Rational Arithmetic:** Why floating-point approximations invalidate finite topological and duality proofs.
2. **Primal-Dual Linear Programming Formulation:** The exact primal standard form over split variables $(r^+, r^-, f^+, f^-)$ and its capacity-bounded circulation dual (Theorems L1C-T1, L1C-T3).
3. **Two-Phase Simplex Walkthrough with Bland's Rule:** Full tableau inspection, Phase 1 artificial variable elimination, and degeneracy-proof pivoting.
4. **Complementary Slackness Certification:** The exact rational certificate $(r^*, z^*)$ satisfying coordinate-level sign matching and zero duality gap $\Delta = 0$ (Theorem L1C-T4).
5. **Simplicial Physics & Discrete Bianchi Identities:** Boundary nilpotency $D_1 D_2 = 0$, field strength vanishing $d_1(d_0 \phi) = 0$ (Theorem L1C-T7), and current conservation $\mathbf{B}^\top \mathbf{z} = 0$.
6. **Finite Quantum Obstruction Calculus:** Exact rational density matrices, positive semidefiniteness, dephasing, and off-diagonal coherence bounds (Theorem L1C-T8).
7. **Formal Lean 4 Mechanization Cross-Reference:** Mapping from prose propositions to the specified finite integer lemmas and the conditional general finite-list certificate in `lean4/`.

In [1]:
import sys
from pathlib import Path
from fractions import Fraction
import numpy as np

# Locate this standalone paper, from its root or notebook directory.
paper_root = Path.cwd()
if paper_root.name == "jupyter_notebooks":
    paper_root = paper_root.parent
found = paper_root / "code"
if not (found / "l1_cohomology.py").is_file():
    raise RuntimeError("Run this notebook from the paper root or jupyter_notebooks directory")
if str(found) not in sys.path:
    sys.path.insert(0, str(found))

from l1_cohomology import (
    DirectedGraph,
    Simplicial2Complex,
    ExactTwoPhaseSimplexSolver,
    L1QuotientSolver,
    L1CertificateReceipt,
    RationalComplex,
    QuantumDensityMatrix,
    ZeroWeightRefusalError,
    DualityGapNonZeroError,
    NilpotencyViolationError,
    NegativeSpectrumError,
    SolverIterationLimitError,
    PrimalFeasibilityError,
    check_certificate_receipt,
)

print("[OK] Pure rational formal environment initialized.")
print("[OK] Code directory linked.")

[OK] Pure rational formal environment initialized.
[OK] Code directory linked.


## Section 1: Exact Rational Arithmetic & The Zero-Float Law

### Why Float Arithmetic Fails Formal Discrete Topology
In continuum analysis, numerical solvers rely on floating-point approximations with arbitrary epsilon tolerances (e.g., `abs(x) < 1e-9`). In algebraic topology and exact duality, this practice introduces severe mathematical defects:
1. **False Duality Gaps:** A true zero duality gap $\Delta = 0$ can appear as $\Delta \approx 10^{-14} \ne 0$, falsely failing complementary slackness certificates.
2. **Broken Nilpotency:** The fundamental identity $D_1 D_2 = 0$ or $d_1(d_0 \phi) = 0$ can produce non-zero rounding residues (e.g. $10^{-16}$), masquerading as fictitious topological curvature.
3. **Simplex Cycling:** In degenerate linear programs, floating-point tie-breaking can cause simplex loops to cycle indefinitely or make invalid pivot selections.

### The Zero-Float Law
Under the Zero-Float Law, every decision path, matrix inversion, Gaussian elimination step, and linear program in this package is executed using standard library `fractions.Fraction`. All arithmetic operations are closed over the field of rational numbers $\mathbb{Q}$, yielding mathematically exact integers and irreducible fractions with zero tolerances.

Let us demonstrate the precision divergence between IEEE-754 double precision floats and exact `Fraction` arithmetic on an ill-conditioned harmonic accumulation.

In [2]:
# Demonstration of Zero-Float Law vs Float Representation Defect
# In IEEE-754 binary floating point, 0.1 and 0.2 cannot be represented exactly:
float_lhs = 0.1 + 0.2
float_rhs = 0.3
float_diff = float_lhs - float_rhs

# In exact rational arithmetic:
frac_lhs = Fraction(1, 10) + Fraction(2, 10)
frac_rhs = Fraction(3, 10)
frac_diff = frac_lhs - frac_rhs

print(f"Exact Rational: (1/10) + (2/10) == (3/10): {frac_lhs == frac_rhs}")
print(f"Exact Difference: {frac_diff} (Zero-Float Law holds!)")
print(f"\nFloat Expression: 0.1 + 0.2 = {float_lhs:.17f}")
print(f"Float Expected:   0.3       = {float_rhs:.17f}")
print(f"Float Difference (0.1 + 0.2 - 0.3): {float_diff:.2e} (Defect in IEEE-754!)")

assert frac_diff == Fraction(0), "Exact rational arithmetic must have zero error!"
assert float_diff != 0.0, "Float arithmetic fails 0.1 + 0.2 == 0.3 in IEEE-754!"
print("\n[OK] Zero-Float Law verified: pure rational arithmetic eliminates all numerical artifacts.")

Exact Rational: (1/10) + (2/10) == (3/10): True
Exact Difference: 0 (Zero-Float Law holds!)

Float Expression: 0.1 + 0.2 = 0.30000000000000004
Float Expected:   0.3       = 0.29999999999999999
Float Difference (0.1 + 0.2 - 0.3): 5.55e-17 (Defect in IEEE-754!)

[OK] Zero-Float Law verified: pure rational arithmetic eliminates all numerical artifacts.


## Section 2: Primal-Dual LP Formulation of L1 Quotient Cohomology

Let $G = (V, E)$ be a finite directed graph with strictly positive weights $\mathbf{w} \in \mathbb{Q}_{>0}^E$ and an observed 1-cochain $\boldsymbol{\omega} \in \mathbb{Q}^E$.

### 1. The Primal Linear Program (Standard Form)
The quotient norm $\Phi_1([\omega]) = \inf_f \|\omega + d_0 f\|_{1, \mathbf{w}}$ is converted into a standard-form linear program by decomposing the residual $r_e = \omega_e + (d_0 f)_e$ into non-negative parts $r_e = r_e^+ - r_e^-$, and decomposing the potential $f_v = f_v^+ - f_v^-$ on all non-gauge vertices:

$$\min_{\mathbf{r}^+, \mathbf{r}^-, \mathbf{f}^+, \mathbf{f}^-} \sum_{e \in E} w_e (r_e^+ + r_e^-)$$

subject to:
$$r_e^+ - r_e^- - \sum_{v \in V \setminus \{v_0\}} \mathbf{B}_{e, v}(f_v^+ - f_v^-) = \omega_e \quad \forall e \in E,$$
$$r_e^+, r_e^- \ge 0, \quad f_v^+, f_v^- \ge 0, \quad f(v_0) = 0 \text{ (gauge root fixed)}.$$

### 2. The Dual Linear Program (Theorem L1C-T3)
By linear programming duality, assigning dual variables $z_e$ to the $m$ equality constraints yields the dual program:

$$\max_{\mathbf{z} \in \mathbb{Q}^E} \mathbf{z}^\top \boldsymbol{\omega} = \sum_{e \in E} z_e \omega_e$$

subject to:
1. **Circulation Law (Kirchhoff Current Law):**
   $$\mathbf{B}^\top \mathbf{z} = 0 \iff \sum_{e: \text{head}(e)=v} z_e = \sum_{e: \text{tail}(e)=v} z_e \quad \forall v \in V.$$
2. **Capacity Bounds:**
   $$-w_e \le z_e \le w_e \quad \forall e \in E.$$

### Weak Duality Theorem
For **any** circulation $\mathbf{z}$ satisfying $\mathbf{B}^\top \mathbf{z} = 0$ and $|z_e| \le w_e$, and **any** potential $\mathbf{f}$:
$$\mathbf{z}^\top \boldsymbol{\omega} = \mathbf{z}^\top (\mathbf{r} - \mathbf{B}\mathbf{f}) = \mathbf{z}^\top \mathbf{r} - (\mathbf{B}^\top \mathbf{z})^\top \mathbf{f} = \mathbf{z}^\top \mathbf{r} = \sum_e z_e r_e \le \sum_e |z_e| |r_e| \le \sum_e w_e |r_e| = \|\mathbf{r}\|_{1, \mathbf{w}}.$$

Therefore, every feasible circulation provides a certified lower bound on $\Phi_1([\omega])$. By strong LP duality, equality holds at the optimum.

In [3]:
# Verify Weak Duality: <z, omega> <= ||r||_{1, w} for all feasible circulations
g_test = DirectedGraph(
    vertices=[0, 1, 2, 3],
    edges=[(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)],
    weights=[Fraction(2), Fraction(3), Fraction(1), Fraction(4), Fraction(5)]
)
omega_test = [Fraction(7), Fraction(-3), Fraction(4), Fraction(2), Fraction(1)]

# Solve primal problem exactly
receipt = L1QuotientSolver.solve(g_test, omega_test)
opt_val = receipt.primal_norm

print(f"Optimal Primal Norm Phi1: {opt_val}")
print(f"Optimal Dual Value:       {receipt.dual_bound}")
print(f"Certified Duality Gap:    {receipt.duality_gap}")

# Generate a family of sub-optimal test circulations by scaling cycle vectors
# Cycle 1: e0 + e1 - e4 (0->1->2->0). Cycle 2: e4 + e2 + e3 (0->2->3->0).
# z_test = alpha * cycle1 + beta * cycle2, scaled to satisfy |z_e| <= w_e.
z_star = receipt.dual_circulation
assert g_test.inner_product(z_star, omega_test) == opt_val

# Check divergence of optimal circulation
div = g_test.d0_star(z_star)
assert all(d == Fraction(0) for d in div), "Optimal dual must be a circulation!"

print(f"\nOptimal Dual Circulation z*:")
for i, ze in enumerate(z_star):
    print(f"  e{i} {g_test.edges[i]}: z_e = {ze}, weight w_e = {g_test.weights[i]}")

print("\n[OK] Strong LP duality verified: exact rational equality and zero duality gap.")

Optimal Primal Norm Phi1: 13
Optimal Dual Value:       13
Certified Duality Gap:    0

Optimal Dual Circulation z*:
  e0 (0, 1): z_e = 2, weight w_e = 2
  e1 (1, 2): z_e = 2, weight w_e = 3
  e2 (2, 3): z_e = 1, weight w_e = 1
  e3 (3, 0): z_e = 1, weight w_e = 4
  e4 (0, 2): z_e = -1, weight w_e = 5

[OK] Strong LP duality verified: exact rational equality and zero duality gap.


## Section 3: Step-by-Step Two-Phase Simplex Walkthrough with Bland's Rule

### The Architecture of `ExactTwoPhaseSimplexSolver`
Standard solvers implement floating-point LU-factorization. Our pure rational solver operates on the standard equality form:
$$\min \mathbf{c}^\top \mathbf{x} \quad \text{s.t.} \quad \mathbf{A}\mathbf{x} = \mathbf{b}, \quad \mathbf{x} \ge 0, \quad \mathbf{b} \ge 0.$$

#### Phase 1: Artificial Variable Elimination
To find an initial Basic Feasible Solution (BFS):
1. Augment the constraint matrix with $m$ artificial variables: $\mathbf{A}\mathbf{x} + \mathbf{a} = \mathbf{b}$, where $\mathbf{a} \ge 0$.
2. Form the Phase 1 objective: minimize $W = \sum_{i=1}^m a_i$.
3. Drive $W$ to 0. If $\min W > 0$, the linear program is mathematically **infeasible**.
4. If $W = 0$, all artificial variables are driven out of the basis (or made non-basic via zero-pivot rotations).

#### Phase 2: Optimizing the Target Cost Vector
With a certified initial BFS in hand, substitute the original cost vector $\mathbf{c}$ into row 0 and pivot until all reduced costs $\bar{c}_j \ge 0$.

#### Bland's Minimal-Index Rule
Degeneracy (multiple bases representing the same geometric vertex) can cause naive simplex implementations to enter infinite cycling loops. To guarantee finite termination:
- **Entering variable:** Choose the non-basic variable with the **smallest column index** $j$ such that $\bar{c}_j < 0$.
- **Leaving variable:** Among rows attaining the minimum ratio test $\theta = b_i / A_{i, j}$, choose the basic variable with the **smallest variable index**.

Let us walk through an explicit simplex solve, printing the exact rational tableau.

In [4]:
# Walkthrough of ExactTwoPhaseSimplexSolver on a clean 2D LP
# Minimize: c^T x = -2 x0 - 3 x1
# Subject to:
#   x0 + 2 x1 + s0       = 8   (e.g., constraint 1)
# 2 x0 +   x1      + s1  = 8   (e.g., constraint 2)
#   x0, x1, s0, s1 >= 0

c = [Fraction(-2), Fraction(-3), Fraction(0), Fraction(0)]
A = [
    [Fraction(1), Fraction(2), Fraction(1), Fraction(0)],
    [Fraction(2), Fraction(1), Fraction(0), Fraction(1)],
]
b = [Fraction(8), Fraction(8)]

res = ExactTwoPhaseSimplexSolver.solve_standard_lp(c, A, b)

print("Linear Program Solved with Exact Rational Simplex (Bland's Rule):")
print(f"  Optimal Status:   {res.status}")
print(f"  Optimal Value:    {res.obj_val} (min -2*x0 - 3*x1 = -40/3)")
print(f"  Solution Vector x:")
print(f"    x0 = {res.x[0]}")
print(f"    x1 = {res.x[1]}")
print(f"    s0 = {res.x[2]}")
print(f"    s1 = {res.x[3]}")

# Verify exact feasibility: A x == b
for i in range(len(A)):
    lhs = sum(A[i][j] * res.x[j] for j in range(len(c)))
    assert lhs == b[i], f"Constraint {i} violated!"
print("[OK] Exact primal feasibility A x == b verified with 0 error.")

Linear Program Solved with Exact Rational Simplex (Bland's Rule):
  Optimal Status:   OPTIMAL
  Optimal Value:    -40/3 (min -2*x0 - 3*x1 = -40/3)
  Solution Vector x:
    x0 = 8/3
    x1 = 8/3
    s0 = 0
    s1 = 0
[OK] Exact primal feasibility A x == b verified with 0 error.


## Section 4: Complementary Slackness Certification & Zero Duality Gap

### Theorem L1C-T4 (Primal-Dual Certificate)
Let $\mathbf{r}^* = \boldsymbol{\omega} + d_0 \mathbf{f}^* \in \mathbb{Q}^E$ and $\mathbf{z}^* \in \mathbb{Q}^E$ be feasible primal and dual solutions:
1. **Primal Feasibility:** $\mathbf{r}^* - d_0 \mathbf{f}^* = \boldsymbol{\omega}$.
2. **Dual Feasibility:** $\mathbf{B}^\top \mathbf{z}^* = 0$ (circulation) and $|z_e^*| \le w_e$ for all $e \in E$.
3. **Complementary Slackness Conditions:**
   - For every edge $e \in \text{supp}(\mathbf{r}^*)$ (where $r_e^* \ne 0$):
     $$z_e^* = w_e \cdot \text{sgn}(r_e^*).$$
   - For every edge $e \notin \text{supp}(\mathbf{r}^*)$ (where $r_e^* = 0$):
     $$-w_e \le z_e^* \le w_e.$$

When these conditions hold:
$$\langle \mathbf{z}^*, \mathbf{r}^*\rangle = \sum_{e \in E} z_e^* r_e^* = \sum_{e \in \text{supp}(\mathbf{r}^*)} w_e \cdot \text{sgn}(r_e^*) r_e^* = \sum_{e \in E} w_e |r_e^*| = \|\mathbf{r}^*\|_{1, \mathbf{w}}.$$
Since $\mathbf{z}^*$ is a circulation, $\langle \mathbf{z}^*, \boldsymbol{\omega}\rangle = \langle \mathbf{z}^*, \mathbf{r}^*\rangle$. Thus the duality gap vanishes identically:
$$\Delta = \|\mathbf{r}^*\|_{1, \mathbf{w}} - \mathbf{z}^{*\top} \boldsymbol{\omega} = 0.$$

Both $\mathbf{r}^*$ and $\mathbf{z}^*$ serve as mutual certificates of optimality. Changing either the graph topology or the weight vector invalidates the certificate.

Let us inspect an emitted `L1CertificateReceipt` and run the exhaustive certificate verification checks.

In [5]:
# Test Certificate Verification on a 5-Vertex Cycle Graph
g_cycle = DirectedGraph.cycle(5, weights=[2, 3, 1, 4, 5])
omega_cycle = [Fraction(10), Fraction(-4), Fraction(7), Fraction(1), Fraction(-2)]

# Solve and obtain certificate receipt
receipt_cycle = L1QuotientSolver.solve(g_cycle, omega_cycle)

print("L1 Certificate Receipt:")
print(f"  Primal Norm:              {receipt_cycle.primal_norm}")
print(f"  Dual Bound:               {receipt_cycle.dual_bound}")
print(f"  Duality Gap:              {receipt_cycle.duality_gap} (must be 0)")
print(f"  Support Size:             {receipt_cycle.support_size} (cycle rank beta = {receipt_cycle.cycle_rank})")
print(f"  Support Bound Satisfied:  {receipt_cycle.satisfies_support_bound}")
print(f"  Complementary Slackness:  {receipt_cycle.satisfies_complementary_slackness}")
print(f"  Is Integral:              {receipt_cycle.is_integral}")
print(f"  Diagnostic Type:          {receipt_cycle.diagnostic_type}")

# Verify coordinate-level complementary slackness conditions
r = receipt_cycle.primal_residual
z = receipt_cycle.dual_circulation
w = g_cycle.weights

cs_verified = True
print("\nCoordinate-by-Coordinate Complementary Slackness Audit:")
for e in range(len(g_cycle.edges)):
    if r[e] > Fraction(0):
        expected_z = w[e]
        status = "PASS" if z[e] == expected_z else "FAIL"
    elif r[e] < Fraction(0):
        expected_z = -w[e]
        status = "PASS" if z[e] == expected_z else "FAIL"
    else:
        expected_z = "[-w_e, +w_e]"
        status = "PASS" if abs(z[e]) <= w[e] else "FAIL"
    
    if "FAIL" in status:
        cs_verified = False
    print(f"  Edge {e} {g_cycle.edges[e]}: r_e = {str(r[e]):>5}, z_e = {str(z[e]):>5}, w_e = {str(w[e]):>2} | [{status}] (expected {expected_z})")

assert cs_verified, "Complementary slackness failed!"
assert receipt_cycle.duality_gap == Fraction(0), "Duality gap must be exactly 0!"
print("\n[OK] Complementary slackness and zero duality gap certified exactly.")

L1 Certificate Receipt:
  Primal Norm:              12
  Dual Bound:               12
  Duality Gap:              0 (must be 0)
  Support Size:             1 (cycle rank beta = 1)
  Support Bound Satisfied:  True
  Complementary Slackness:  True
  Is Integral:              True
  Diagnostic Type:          ClosedResidual

Coordinate-by-Coordinate Complementary Slackness Audit:
  Edge 0 (0, 1): r_e =     0, z_e =     1, w_e =  2 | [PASS] (expected [-w_e, +w_e])
  Edge 1 (1, 2): r_e =     0, z_e =     1, w_e =  3 | [PASS] (expected [-w_e, +w_e])
  Edge 2 (2, 3): r_e =    12, z_e =     1, w_e =  1 | [PASS] (expected 1)
  Edge 3 (3, 4): r_e =     0, z_e =     1, w_e =  4 | [PASS] (expected [-w_e, +w_e])
  Edge 4 (4, 0): r_e =     0, z_e =     1, w_e =  5 | [PASS] (expected [-w_e, +w_e])

[OK] Complementary slackness and zero duality gap certified exactly.


## Section 5: Discrete Physics & Simplicial Bianchi Identities

### The Simplicial de Rham Complex
Let $K = (V, E, F)$ be an oriented 2-dimensional simplicial complex.
We have the chain complex:
$$C_2 \xrightarrow{D_2} C_1 \xrightarrow{D_1} C_0,$$
and its dual cochain complex:
$$C^0 \xrightarrow{d_0 = D_1^\top} C^1 \xrightarrow{d_1 = D_2^\top} C^2.$$

#### 1. Exact Boundary Nilpotency
The topological boundary of a boundary is empty:
$$D_1 \circ D_2 = 0.$$
Each triangular face $f = (u, v, w)$ has boundary cycle $\partial f = (u, v) + (v, w) - (u, w)$. Applying vertex boundary $D_1$:
$$D_1(\partial f) = (v - u) + (w - v) - (w - u) = 0.$$

#### 2. Discrete Bianchi Identity (Theorem L1C-T7)
Let $\phi \in C^0$ be a discrete 0-cochain (lattice gauge potential).
The gauge connection (vector potential) is $A = d_0 \phi \in C^1$.
The discrete field strength (curvature tensor) through faces is $F = d_1 A = d_1 (d_0 \phi) \in C^2$.
Using the duality relation $d_0 = D_1^\top$ and $d_1 = D_2^\top$:
$$d_1(d_0 \phi) = D_2^\top (D_1^\top \phi) = (D_1 D_2)^\top \phi = \mathbf{0}^\top \phi = 0.$$

#### 3. Discrete Gauss's Law / Current Conservation
The dual circulation $\mathbf{z}^*$ satisfies $\mathbf{B}^\top \mathbf{z}^* = D_1 \mathbf{z}^* = 0$. In lattice electromagnetism, $\mathbf{z}^*$ represents a divergence-free steady-state current density satisfying the discrete continuity equation $\nabla \cdot \mathbf{j} = 0$.

Let us verify the boundary matrices and Bianchi identity with exact rational matrix multiplication.

In [6]:
# Verification of D1, D2, and Bianchi Nilpotency on 2-Complex
sc_oct = Simplicial2Complex(
    vertices=[0, 1, 2, 3, 4],
    edges=[(0, 1), (1, 2), (0, 2), (2, 3), (0, 3), (3, 4), (0, 4)],
    faces=[(0, 1, 2), (0, 2, 3), (0, 3, 4)],
    weights=[1, 1, 1, 1, 1, 1, 1]
)

print("Boundary Matrix D1 (|V| x |E| = 5 x 7):")
for r in sc_oct.D1:
    print(f"  {[int(x) for x in r]}")

print("\nBoundary Matrix D2 (|E| x |F| = 7 x 3):")
for r in sc_oct.D2:
    print(f"  {[int(x) for x in r]}")

# Compute product D1 @ D2 over Fraction
prod_D1_D2 = [
    [sum(sc_oct.D1[v][e] * sc_oct.D2[e][f] for e in range(len(sc_oct.edges))) for f in range(len(sc_oct.faces))]
    for v in range(len(sc_oct.vertices))
]

print("\nProduct D1 @ D2 (|V| x |F| = 5 x 3):")
all_zero = True
for r in prod_D1_D2:
    print(f"  {[int(x) for x in r]}")
    if any(x != Fraction(0) for x in r):
        all_zero = False

assert all_zero, "D1 @ D2 != 0! Topological boundary violation!"
print("\n[OK] Topological nilpotency D1 @ D2 == 0 mathematically verified.")

# Verify field strength F = d1(d0 phi) vanishes on all faces for arbitrary phi
phi = [Fraction(17), Fraction(-8), Fraction(99), Fraction(-4), Fraction(31)]
d0_phi = sc_oct.d0(phi)
d1_d0_phi = sc_oct.discrete_bianchi_identity(phi)

print(f"\nGauge potential phi:          {[int(x) for x in phi]}")
print(f"Gauge connection A = d0(phi): {[int(x) for x in d0_phi]}")
print(f"Field strength F = d1(d0 phi):{[int(x) for x in d1_d0_phi]} (identically zero)")
print("[OK] Discrete Bianchi Identity (Theorem L1C-T7) certified.")

Boundary Matrix D1 (|V| x |E| = 5 x 7):
  [-1, 0, -1, 0, -1, 0, -1]
  [1, -1, 0, 0, 0, 0, 0]
  [0, 1, 1, -1, 0, 0, 0]
  [0, 0, 0, 1, 1, -1, 0]
  [0, 0, 0, 0, 0, 1, 1]

Boundary Matrix D2 (|E| x |F| = 7 x 3):
  [1, 0, 0]
  [1, 0, 0]
  [-1, 1, 0]
  [0, 1, 0]
  [0, -1, 1]
  [0, 0, 1]
  [0, 0, -1]

Product D1 @ D2 (|V| x |F| = 5 x 3):
  [0, 0, 0]
  [0, 0, 0]
  [0, 0, 0]
  [0, 0, 0]
  [0, 0, 0]

[OK] Topological nilpotency D1 @ D2 == 0 mathematically verified.

Gauge potential phi:          [17, -8, 99, -4, 31]
Gauge connection A = d0(phi): [-25, 107, 82, -103, -21, 35, 14]
Field strength F = d1(d0 phi):[0, 0, 0] (identically zero)
[OK] Discrete Bianchi Identity (Theorem L1C-T7) certified.


## Section 6: Quantum Phase Lifts and Trace-Norm Coherence

For rational complex entries, the implementation checks Hermiticity, unit trace and PSD exactly by rational $LDL^\dagger$ elimination; two-by-two minors alone are insufficient in dimension at least three. For a fixed density matrix, the classical trace-distance bound is
$$\frac12\operatorname{Tr}|\rho-\rho_{\mathrm{diag}}|\le\sum_{u<v}|\rho_{uv}|=\frac12 C_{l_1}(\rho).$$
The modulus is exact only when the required square root is rational; otherwise the displayed coherence is binary64.

Phase lifts belong to $\mathbb{R}^{E_\rho}$ on the support graph $E_\rho=\{(u,v):\rho_{uv}\ne0\}$. Conjugating by $U_\theta=\operatorname{diag}(e^{-i\theta_v})$ adds $d_0\theta$. The real quotient norm is invariant under these real gradient shifts, while principal arguments can add $2\pi k$. Branch-independent multiplicative holonomy is defined on integral cycles $Z_1(G_\rho;\mathbb Z)$; real circulations remain the domain of the LP dual. The period-minimized scalar is a distance from the identity in the quotient group, not a homogeneous real norm. These statements concern static finite geometry, not a dynamical coherence-survival theorem.


In [7]:
# Evaluate exact rational quantum density matrix over 4 vertices
# Pure state vector |psi> = 1/2 |0> + 1/2 |1> + 1/2 |2> + 1/2 |3>
# Sum of squares = 4 * (1/4) = 1.
psi_amplitudes = [
    RationalComplex(Fraction(1, 2), Fraction(0)),
    RationalComplex(Fraction(1, 2), Fraction(0)),
    RationalComplex(Fraction(1, 2), Fraction(0)),
    RationalComplex(Fraction(1, 2), Fraction(0)),
]

rho_4 = QuantumDensityMatrix.pure_state(psi_amplitudes)
print(f"Density Matrix Dimension: {rho_4.dim} x {rho_4.dim}")

# Validate state via exact rational LDL^dagger decomposition
rho_4.validate_state()
print("Exact rational LDL^dagger positive-semidefiniteness certified.")

print("\nMatrix Elements rho_{jk}:")
for j in range(rho_4.dim):
    row_re = [f"{rho_4.matrix[j][k].re}" for k in range(rho_4.dim)]
    print(f"  Row {j}: {row_re}")

# Verify 3x3 non-PSD counterexample is properly rejected
bad_mat = [
    [RationalComplex(Fraction(4, 12), Fraction(0)), RationalComplex(Fraction(-3, 12), Fraction(0)), RationalComplex(Fraction(-3, 12), Fraction(0))],
    [RationalComplex(Fraction(-3, 12), Fraction(0)), RationalComplex(Fraction(4, 12), Fraction(0)), RationalComplex(Fraction(-3, 12), Fraction(0))],
    [RationalComplex(Fraction(-3, 12), Fraction(0)), RationalComplex(Fraction(-3, 12), Fraction(0)), RationalComplex(Fraction(4, 12), Fraction(0))],
]
try:
    rho_bad = QuantumDensityMatrix(bad_mat)
    assert False, "Non-PSD matrix should have been rejected!"
except (ValueError, NegativeSpectrumError) as e:
    print(f"\n3x3 Non-PSD Counterexample correctly rejected: {e}")

# Verify complex off-diagonal coherence with true modulus
plus_i_state = [
    [RationalComplex(Fraction(1, 2), Fraction(0)), RationalComplex(Fraction(0), Fraction(1, 2))],
    [RationalComplex(Fraction(0), Fraction(-1, 2)), RationalComplex(Fraction(1, 2), Fraction(0))]
]
rho_complex = QuantumDensityMatrix(plus_i_state)
rho_complex.validate_state()
c_off = rho_complex.off_diagonal_l1_coherence()
print(f"Complex state |+i><+i| off-diagonal coherence: {c_off} (must be 1/2, not 0)")
assert c_off == Fraction(1, 2), f"Expected 1/2, got {c_off}"

# Dephased diagonal state on 4-vertex pure state
rho_diag = rho_4.dephased_state()
l1_coherence = rho_4.off_diagonal_l1_coherence()
total_off_diag = rho_4.total_l1_coherence()
print(f"\n4-Vertex State Off-Diagonal Coherence sum_{{j < k}} |rho_jk|: {l1_coherence}")
print(f"Total Off-Diagonal Coherence C_{{l_1}}(rho): {total_off_diag}")
print(f"Purity Tr(rho^2): {sum((rho_4.matrix[j][k].norm_sq() for j in range(rho_4.dim) for k in range(rho_4.dim)), Fraction(0))} (pure state)")
print("[OK] Quantum Density Matrix constraints and trace coherence certified.")

Density Matrix Dimension: 4 x 4
Exact rational LDL^dagger positive-semidefiniteness certified.

Matrix Elements rho_{jk}:
  Row 0: ['1/4', '1/4', '1/4', '1/4']
  Row 1: ['1/4', '1/4', '1/4', '1/4']
  Row 2: ['1/4', '1/4', '1/4', '1/4']
  Row 3: ['1/4', '1/4', '1/4', '1/4']

3x3 Non-PSD Counterexample correctly rejected: [L1_COHOMOLOGY_ERROR] State violates positive semidefiniteness (has negative eigenvalues)
Complex state |+i><+i| off-diagonal coherence: 1/2 (must be 1/2, not 0)

4-Vertex State Off-Diagonal Coherence sum_{j < k} |rho_jk|: 3/2
Total Off-Diagonal Coherence C_{l_1}(rho): 3
Purity Tr(rho^2): 1 (pure state)
[OK] Quantum Density Matrix constraints and trace coherence certified.


## Section 7: Formal Mechanization Cross-Reference

Paper 002 is accompanied by an offline-compiling **Lean 4 formal verification estate** located in `lean4/`.
The suite checks specified integer finite-model lemmas and the conditional finite-list certificate under proof-assistant kernel checking with axioms whitelist `{propext, Classical.choice, Quot.sound}` and zero `sorry` placeholders.

### Theorem-to-Module Mapping Table

| Claim ID | Mathematical Statement | Paper Section | Status | Lean 4 Lake Module | Mechanized Theorem Names |
|:---|:---|:---:|:---:|:---|:---|
| **L1C-T1** | Integer three-cycle gauge-fixed coercivity bound | §2 | `MECHANIZED` | `l1c_t01_attainment.lean` | `gauge_fixed_kernel_trivial`, `potential_v1_le_edge0`, `potential_v2_le_edge_sum`, `l1c_t01_coercivity_bound` |
| **L1C-T2** | Integer three-edge ambient weighted-norm axioms and residual addition | §2 | `MECHANIZED` | `l1c_t02_quotient_norm.lean` | `cycle3_d0_add`, `residual3_add`, `l1_norm3_nonneg`, `l1_norm3_triangle`, `l1_quotient_subadditive`, `l1_norm3_positive_definite`, `zero_weight_refusal_counterexample` |
| **L1C-T3** | Integer three-cycle weak-duality inequality | §3 | `MECHANIZED` | `l1c_t03_weak_duality.lean` | `int_mul_le_weight_abs`, `circulation_annihilates_coboundary`, `circulation_pairing_gauge_invariant`, `l1c_t03_weak_duality` |
| **L1C-C1** | Exactness criterion by circulation pairings | §3 | `PROSE` | `l1c_t03_weak_duality.lean` | Corollary of L1C-T2 and L1C-T3; checked algebraically |
| **L1C-T4** | Complementary-slackness certificate & zero duality gap | §4 | `MECHANIZED` | `l1c_t04_complementary_slackness.lean` | `pairing_eq_l1_norm_of_cs`, `l1c_t04_zero_duality_gap`, `l1c_t04_primal_optimality` |
| **L1C-TG** | Conditional integer finite-list certificate optimality | §4 | `MECHANIZED` | `l1c_general_certificate.lean` | `dot_vec_add`, `dot_le_weighted_l1_norm`, `general_finite_certificate_optimality` |
| **L1C-T5** | Integer three-cycle one-sparse attainment witness | §5 | `MECHANIZED` | `l1c_t05_cycle_invariant.lean` | `cycle3_residual_flux_invariant`, `abs_sum3_le`, `nat_weighted_sum_lower_bound`, `cycle3_topological_lower_bound`, `sparse_representative_attainment` |
| **L1C-T6** | Integral data admit integral optimum | §6 | `PROSE` | Computational | Total unimodularity of deleted incidence matrix (Dey et al. 2011) |
| **L1C-T7** | Triangle, plaquette and five-cycle Bianchi witnesses | §1.4 / §7 | `MECHANIZED` | `l1c_t07_bianchi_nilpotency.lean` | `triangle_bianchi_nilpotency`, `plaquette_bianchi_nilpotency`, `cycle5_loop_nilpotency` |
| **L1C-T8** | Quantum Phase Cocycle & Trace Defect Bound | §1.5 / §8 | `PROSE` | Computational | Trace-distance off-diagonal coherence bounding |
| **L1C-NC1**| Nonzero classes are `ClosedResidual` (not stabilized) | §9 | `SCOPE FENCE` | None | Definitional boundary preventing speculative overclaim |

Let us run an automated inspection script that scans the local `lean4/` directory and audits the proof files.
The ambient norm lemmas do not define a minimized quotient norm. General results have prose proofs and are classical background. See `docs/lean_claim_map.md` for exact coverage.


In [8]:
# Audit the Lean 4 formalization estate in lean4/
import re

lean4_dir = None
for base in [Path.cwd(), Path.cwd().parent, *Path.cwd().parents]:
    cand = base / "coda_atlas" / "F004_papers" / "001_finite_obstruction_calculus" / "007_l1_native_cohomology" / "lean4"
    if cand.exists():
        lean4_dir = cand
        break
    if (base / "lean4").exists():
        lean4_dir = base / "lean4"
        break

print("Lean 4 Estate Directory located.")

lean_files = sorted([f for f in lean4_dir.glob("*.lean") if f.name not in ("lakefile.lean", "Census.lean")])
print(f"Found {len(lean_files)} theorem modules:\n")

total_lines = 0
total_theorems = 0
sorry_count = 0

for lf in lean_files:
    raw_content = lf.read_text(encoding="utf-8")
    # Strip block comments /- ... -/ and line comments -- ...
    no_block_comments = re.sub(r"/-\!.*?-\/", "", raw_content, flags=re.DOTALL)
    no_block_comments = re.sub(r"/-.*?-\/", "", no_block_comments, flags=re.DOTALL)
    code_lines = []
    for line in no_block_comments.splitlines():
        line_clean = re.sub(r"--.*$", "", line).strip()
        if line_clean:
            code_lines.append(line_clean)
            
    lines = raw_content.splitlines()
    total_lines += len(lines)
    theorems = [l for l in code_lines if l.startswith("theorem") or l.startswith("def") or l.startswith("lemma")]
    total_theorems += len(theorems)
    # Check for actual sorry keywords in code (not in comments)
    sorries = [l for l in code_lines if re.search(r"\bsorry\b", l)]
    sorry_count += len(sorries)
    
    print(f"  {lf.name:<36} | {len(lines):4} lines | {len(theorems):2} theorems/defs | sorries: {len(sorries)}")

print(f"\nTotal Formal Estate: {total_lines} lines across {len(lean_files)} theorem modules.")
print(f"Total Mechanized Theorems / Definitions: {total_theorems}")
print(f"Total Unproven 'sorry' Proof Stubs:     {sorry_count} (zero tolerance met)")

assert sorry_count == 0, "Formal estate contains unproven sorry stubs!"
print("\n[OK] Formal Lean 4 estate verified: 0 sorries, source text audit complete.")

Lean 4 Estate Directory located.
Found 12 theorem modules:

  l1c_core.lean                        |   35 lines |  0 theorems/defs | sorries: 0
  l1c_general_certificate.lean         |  163 lines |  9 theorems/defs | sorries: 0
  l1c_lp_cochains.lean                 |  154 lines | 25 theorems/defs | sorries: 0
  l1c_lp_duality.lean                  |  101 lines |  8 theorems/defs | sorries: 0
  l1c_lp_hodge_decomposition.lean      |   97 lines |  6 theorems/defs | sorries: 0
  l1c_lp_strict_uniqueness.lean        |  152 lines |  9 theorems/defs | sorries: 0
  l1c_t01_attainment.lean              |  100 lines |  7 theorems/defs | sorries: 0
  l1c_t02_quotient_norm.lean           |  167 lines | 13 theorems/defs | sorries: 0
  l1c_t03_weak_duality.lean            |  136 lines | 11 theorems/defs | sorries: 0
  l1c_t04_complementary_slackness.lean |   66 lines |  4 theorems/defs | sorries: 0
  l1c_t05_cycle_invariant.lean         |   99 lines |  7 theorems/defs | sorries: 0
  l1c_t07_bianch

## Section 8: Conclusion & Claim Hygiene Discipline

In this formal track notebook, we have demonstrated:
1. **The Zero-Float Law:** Exact rational decision checks certify the supplied finite instance; binary64 diagnostics are identified separately.
2. **Primal-Dual Attainment:** The primal L1 quotient problem and its dual capacity-bounded circulation program attain identical optimal objective values with exact zero duality gap $\Delta = 0$ (Theorems L1C-T1, L1C-T3).
3. **Complementary Slackness Certificates:** An explicit rational pair $(\mathbf{r}^*, \mathbf{z}^*)$ certifies mutual optimality through coordinate sign conditions (Theorem L1C-T4).
4. **Discrete Bianchi Nilpotency:** The simplicial exterior derivative satisfies $d_1(d_0 \phi) = 0$ over arbitrary potential fields (Theorem L1C-T7).
5. **Quantum Phase Cocycles:** Static trace-coherence bounds and support-graph phase domains are distinct (Theorem L1C-T8).
6. **Lean 4 Proof Parity:** Lean checks the specified integer finite-model lemmas and conditional finite-list certificate. Prose proves the general finite-dimensional classical results; Lean does not verify the Python implementation or certificate existence.

### Claim Fences (`RULE[user_global]`)
- **What this notebook supports:** Demonstrates that for finite directed graphs and simplicial 2-complexes with positive rational weights, L1-native cohomology is computable via exact two-phase simplex linear programming, attains exact zero duality gaps, emits machine-checkable certificates, and aligns with Lean 4 formalization.
- **What this notebook does NOT support:** It does not prove an L1 Hodge theorem for Riemannian manifolds, does not claim physical selection of L1 norms in continuum quantum electrodynamics, and does not claim stabilization of open residuals.
- **Narrowest defensible claim:** The weighted $L^1$ quotient infimum on finite graphs is an attained norm certified by dual circulations and admits exact rational computation with support bounded by the cycle rank.